**Mount Google Drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Load Cleaned Dataset**

In [2]:
import pandas as pd

file_path = "/content/drive/MyDrive/c01-price-forecasting/data/processed/cleaned_data.csv"
df = pd.read_csv(file_path)

df['date'] = pd.to_datetime(df['date'])

print("Loaded shape:", df.shape)
df.head()

Loaded shape: (2308, 30)


,date,district,paddy_type,min_price,max_price,avg_price,production_total,price_range,week_of_year,week_sin,...,year,month,week,lag_1,lag_2,lag_4,lag_12,rolling_mean_4,rolling_std_4,season_Yala
0,2015-03-26,Ampara,Long_Grain_White,34.0,40.0,37.14,307661,6.0,13,1.000000,...,2015,3,13,37.48,38.50,36.34,27.50,37.6575,0.586195,False
1,2015-04-02,Ampara,Long_Grain_White,34.0,40.0,36.40,309335,6.0,14,0.992709,...,2015,4,14,37.14,37.48,37.51,26.10,37.3800,0.872238,True
2,2015-04-09,Ampara,Long_Grain_White,34.0,39.0,35.68,309335,5.0,15,0.970942,...,2015,4,15,36.40,37.14,38.50,26.58,36.6750,0.802060,True
3,2015-04-16,Ampara,Long_Grain_White,31.0,37.0,34.97,309335,6.0,16,0.935016,...,2015,4,16,35.68,36.40,37.48,28.95,36.0475,0.933430,True
4,2015-04-23,Ampara,Long_Grain_White,30.0,38.0,34.03,309335,8.0,17,0.885456,...,2015,4,17,34.97,35.68,37.14,30.64,35.2700,1.012028,True


**Sort Dataset**

In [3]:
df = df.sort_values(['district', 'date']).reset_index(drop=True)

print("After sorting:")
df[['district','date']].head()

After sorting:


,district,date
0,Ampara,2015-03-26
1,Ampara,2015-04-02
2,Ampara,2015-04-09
3,Ampara,2015-04-16
4,Ampara,2015-04-23


**Train/Test Split (Per District)**

In [4]:
train_parts = []
test_parts  = []

for d in df['district'].unique():
    temp = df[df['district'] == d].sort_values('date')

    split_idx = int(len(temp) * 0.8)

    train_parts.append(temp.iloc[:split_idx])
    test_parts.append(temp.iloc[split_idx:])

train_df = pd.concat(train_parts).reset_index(drop=True)
test_df  = pd.concat(test_parts).reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (1844, 30)
Test shape : (464, 30)


**Validate Split**

In [5]:
# Check district count
print("Train districts:", train_df['district'].nunique())
print("Test districts :", test_df['district'].nunique())

# Check time order per district
for d in df['district'].unique():
    tr_max = train_df[train_df['district']==d]['date'].max()
    te_min = test_df[test_df['district']==d]['date'].min()

    if pd.notna(tr_max) and pd.notna(te_min):
        if tr_max > te_min:
            print(f"Issue in district: {d}")

Train districts: 4
Test districts : 4


**Save Split to Drive**

In [6]:
train_path = "/content/drive/MyDrive/c01-price-forecasting/data/processed/train_data.csv"
test_path  = "/content/drive/MyDrive/c01-price-forecasting/data/processed/test_data.csv"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print("Train/Test saved to Google Drive")

Train/Test saved to Google Drive


**Remove Duplicate Lag Features**

In [7]:
drop_duplicate_lags = [
    'price_t-1','price_t-2','price_t-3',
    'price_t-4','price_t-8','price_t-12'
]

# Apply to both train and test
train_df = train_df.drop(columns=[c for c in drop_duplicate_lags if c in train_df.columns])
test_df  = test_df.drop(columns=[c for c in drop_duplicate_lags if c in test_df.columns])

print("Duplicate lag features removed (if existed)")
print("Remaining columns:", train_df.columns.tolist())

Duplicate lag features removed (if existed)
Remaining columns: ['date', 'district', 'paddy_type', 'min_price', 'max_price', 'avg_price', 'production_total', 'price_range', 'week_of_year', 'week_sin', 'week_cos', 'price_4w_avg', 'price_8w_avg', 'price_change', 'year', 'month', 'week', 'lag_1', 'lag_2', 'lag_4', 'lag_12', 'rolling_mean_4', 'rolling_std_4', 'season_Yala']


**Prepare Data for XGBoost**

In [8]:
# Drop columns not used as features
drop_cols = ['date', 'avg_price']

X_train = train_df.drop(columns=drop_cols)
y_train = train_df['avg_price']

X_test  = test_df.drop(columns=drop_cols)
y_test  = test_df['avg_price']

# Encode categorical variables (ONLY for XGBoost)
X_train = pd.get_dummies(X_train, columns=['district', 'paddy_type'], drop_first=True)
X_test  = pd.get_dummies(X_test,  columns=['district', 'paddy_type'], drop_first=True)

# Align columns (important!)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

X_train shape: (1844, 23)
X_test shape : (464, 23)


**Save XGBoost Data**

In [9]:
import os

base_path = "/content/drive/MyDrive/c01-price-forecasting/data/model_inputs/xgboost/"
os.makedirs(base_path, exist_ok=True)

X_train.to_csv(base_path + "X_train.csv", index=False)
X_test.to_csv(base_path + "X_test.csv", index=False)

y_train.to_csv(base_path + "y_train.csv", index=False)
y_test.to_csv(base_path + "y_test.csv", index=False)

print("XGBoost data saved")

XGBoost data saved


**Scaling (For LSTM / Transformer ONLY)**

In [10]:
from sklearn.preprocessing import MinMaxScaler

# Remove non-numeric columns for DL
exclude_cols = ['date', 'district', 'paddy_type']

feature_cols = [
    c for c in train_df.columns
    if c not in exclude_cols + ['avg_price']
]

print("Features used for DL:", feature_cols)

scaler = MinMaxScaler()

# Fit ONLY on training data
train_vals = train_df[feature_cols + ['avg_price']]
test_vals  = test_df[feature_cols + ['avg_price']]

scaler.fit(train_vals)

train_scaled = pd.DataFrame(
    scaler.transform(train_vals),
    columns=train_vals.columns
)

test_scaled = pd.DataFrame(
    scaler.transform(test_vals),
    columns=test_vals.columns
)

# Keep district + date for grouping later
train_df_scaled = train_df[['district', 'date']].copy()
test_df_scaled  = test_df[['district', 'date']].copy()

train_df_scaled = pd.concat([train_df_scaled, train_scaled], axis=1)
test_df_scaled  = pd.concat([test_df_scaled, test_scaled], axis=1)

print("Scaling completed")

Features used for DL: ['min_price', 'max_price', 'production_total', 'price_range', 'week_of_year', 'week_sin', 'week_cos', 'price_4w_avg', 'price_8w_avg', 'price_change', 'year', 'month', 'week', 'lag_1', 'lag_2', 'lag_4', 'lag_12', 'rolling_mean_4', 'rolling_std_4', 'season_Yala']
Scaling completed


**Save Scaled Data (DL)**

In [11]:
base_path = "/content/drive/MyDrive/c01-price-forecasting/data/model_inputs/lstm/"
os.makedirs(base_path, exist_ok=True)

train_df_scaled.to_csv(base_path + "train_scaled.csv", index=False)
test_df_scaled.to_csv(base_path + "test_scaled.csv", index=False)

print("Scaled data saved")

Scaled data saved


**Save Scaler**

In [12]:
import joblib
import os

scaler_path = "/content/drive/MyDrive/c01-price-forecasting/data/model_inputs/lstm/"
os.makedirs(scaler_path, exist_ok=True)

joblib.dump(scaler, scaler_path + "scaler.pkl")

print("Scaler saved successfully")

Scaler saved successfully


**Create Sequences (LSTM / Transformer)**

In [13]:
import numpy as np

SEQ_LEN = 8  # later tune

def create_sequences(df_part, feature_cols, seq_len=SEQ_LEN):
    X, y = [], []

    values = df_part[feature_cols + ['avg_price']].values

    for i in range(len(values) - seq_len):
        X.append(values[i:i+seq_len, :-1])   # features
        y.append(values[i+seq_len, -1])      # target

    return np.array(X), np.array(y)


def build_sequences(df_input, feature_cols, seq_len=SEQ_LEN):
    X_all, y_all = [], []

    for d in df_input['district'].unique():
        temp = df_input[df_input['district'] == d].sort_values('date')

        if len(temp) <= seq_len:
            continue  # skip small districts

        X, y = create_sequences(temp, feature_cols, seq_len)

        X_all.append(X)
        y_all.append(y)

    return np.vstack(X_all), np.concatenate(y_all)


# Build sequences
X_train_seq, y_train_seq = build_sequences(train_df_scaled, feature_cols)
X_test_seq,  y_test_seq  = build_sequences(test_df_scaled,  feature_cols)

print("Train sequences:", X_train_seq.shape)
print("Test sequences :", X_test_seq.shape)

Train sequences: (1812, 8, 20)
Test sequences : (432, 8, 20)


**Save Sequence Data**

In [14]:
import numpy as np

base_path = "/content/drive/MyDrive/c01-price-forecasting/data/model_inputs/lstm/"

np.save(base_path + "X_train_seq.npy", X_train_seq)
np.save(base_path + "y_train_seq.npy", y_train_seq)

np.save(base_path + "X_test_seq.npy", X_test_seq)
np.save(base_path + "y_test_seq.npy", y_test_seq)

print("Sequence data saved")

Sequence data saved
